<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aibizx/python-primer-notebooks/blob/main/03-pandas.ipynb)

_Part of the [AI/Biz books](https://www.ai.biz/books/python-primer/) collection._

# Chapter 3 — Pandas: The Eighty Per Cent

Companion notebook to [Pandas: The Eighty Per Cent](https://www.ai.biz/books/python-primer/pandas/).

Deliberately messy data, and the specific failures that produce a plausible wrong answer.


In [ ]:
import pandas as pd
import numpy as np
print('pandas', pd.__version__)


## 1. The index is doing more than you think

Pandas aligns on **labels**, not positions. NumPy would have added these positionally.


In [ ]:
a = pd.Series([1, 2, 3], index=['x', 'y', 'z'])
b = pd.Series([10, 20, 30], index=['z', 'y', 'x'])
print(a + b)
print()
print('x is 1 + 30 = 31, matched by label')


## 2. loc is labels, iloc is positions


In [ ]:
df = pd.DataFrame({
    'city': ['Delhi', 'Tokyo', 'Sydney', 'Paris'],
    'population_m': [33.8, 37.1, 5.3, 11.2],
    'region': ['Asia', 'Asia', 'Oceania', 'Europe'],
})
d = df.set_index('city')

print(d.loc['Delhi'].to_dict())
print(d.iloc[0].to_dict())
try:
    d.loc[0]
except KeyError:
    print('d.loc[0] -> KeyError, there is no label 0')


In [ ]:
# loc slices INCLUDE the endpoint. iloc does not.
print('loc[0:2]  ->', len(df.loc[0:2]), 'rows')
print('iloc[0:2] ->', len(df.iloc[0:2]), 'rows')


## 3. Chained assignment — the most common pandas bug


In [ ]:
work = df.copy()

# Wrong: two __getitem__ calls, may assign into a temporary
work[work.region == 'Asia']['population_m'] = 0
print('after chained assignment:')
print(work)

# Right: one .loc call
work.loc[work.region == 'Asia', 'population_m'] = 0
print()
print('after .loc assignment:')
print(work)


## 4. dtypes, and why object is a warning sign


In [ ]:
messy = pd.DataFrame({
    'id': ['0012', '0013', '0014'],
    'amount': ['1,200', '3,400', '900'],
    'region': ['gold', 'gold', 'silver'],
})
print(messy.dtypes)
print()
messy['amount'] = messy['amount'].str.replace(',', '').astype('int64')
messy['region'] = messy['region'].astype('category')
print(messy.dtypes)
print()
print('memory saved on region:',
      pd.Series(['gold','silver']*5000).memory_usage(deep=True),
      'vs',
      pd.Series(['gold','silver']*5000).astype('category').memory_usage(deep=True))


### Leading zeros: the identifier disaster


In [ ]:
import io
csv = 'customer_id,name\n0012345,Asha\n0067890,Ravi\n'

bad = pd.read_csv(io.StringIO(csv))
good = pd.read_csv(io.StringIO(csv), dtype={'customer_id': 'string'})

print('guessed: ', bad.customer_id.tolist(), bad.customer_id.dtype)
print('explicit:', good.customer_id.tolist(), good.customer_id.dtype)
print()
print('The first one will never join to anything again.')


## 5. Missing data


In [ ]:
g = pd.DataFrame({
    'a': [1.0, np.nan, 3.0, 4.0],
    'b': ['x', 'y', None, 'z'],
    'grp': ['p', 'p', None, 'q'],
})
print(g.isna().sum())
print()
print('proportion missing:')
print(g.isna().mean())


In [ ]:
# groupby DROPS missing keys by default. This quietly loses rows.
print('default   :', g.groupby('grp')['a'].count().sum(), 'rows counted')
print('dropna=Fal:', g.groupby('grp', dropna=False)['a'].count().sum(), 'rows counted')


## 6. Split, apply, combine


In [ ]:
print(df.groupby('region')['population_m'].agg(['mean', 'median', 'count']))
print()
# Named aggregation gives flat, readable column names
print(df.groupby('region').agg(
    avg_pop=('population_m', 'mean'),
    n_cities=('city', 'count'),
))


### transform vs agg

`agg` gives one row per group. `transform` gives one row per **original** row.


In [ ]:
out = df.copy()
out['region_avg'] = out.groupby('region')['population_m'].transform('mean')
out['vs_region'] = out['population_m'] - out['region_avg']
print(out)


## 7. The merge that ruins your afternoon

One accidental duplicate on the right silently multiplies your rows. No error, no warning.


In [ ]:
orders = pd.DataFrame({
    'order_id': [1, 2, 3],
    'customer_id': ['A', 'B', 'C'],
    'revenue': [100, 200, 300],
})
customers = pd.DataFrame({
    'customer_id': ['A', 'B', 'C', 'B'],   # <- B appears twice
    'segment': ['smb', 'ent', 'smb', 'ent'],
})

bad = orders.merge(customers, on='customer_id', how='left')
print(f'orders in:  {len(orders)} rows, revenue {orders.revenue.sum()}')
print(f'after join: {len(bad)} rows, revenue {bad.revenue.sum()}  <- inflated')
print(bad)


In [ ]:
# validate= states what you believe and fails immediately if you are wrong
try:
    orders.merge(customers, on='customer_id', how='left', validate='many_to_one')
except pd.errors.MergeError as e:
    print('caught it:', e)


In [ ]:
# The habit worth building
clean = customers.drop_duplicates('customer_id')
before = len(orders)
merged = orders.merge(clean, on='customer_id', how='left',
                      validate='many_to_one', indicator=True)
assert len(merged) == before, f'row count changed: {before} -> {len(merged)}'
print(merged['_merge'].value_counts())
print(f'revenue preserved: {merged.revenue.sum()}')


### Key type mismatch: pandas catches this one for you


In [ ]:
left = pd.DataFrame({'k': [1, 2, 3], 'v': ['a', 'b', 'c']})
right = pd.DataFrame({'k': ['1', '2', '3'], 'w': ['x', 'y', 'z']})

try:
    left.merge(right, on='k')
except ValueError as e:
    print('pandas refuses:', e)

print()
print('after casting ->', len(left.merge(right.astype({'k': 'int64'}), on='k')), 'matches')


## 8. Reshaping


In [ ]:
wide = pd.DataFrame({
    'city': ['Delhi', 'Tokyo'],
    '2023': [33.8, 36.5],
    '2024': [33.4, 35.3],
    '2025': [33.8, 37.1],
})
long = wide.melt(id_vars=['city'], var_name='year', value_name='population_m')
print(long)
print()
print(long.pivot(index='city', columns='year', values='population_m'))


## 9. Method chaining


In [ ]:
result = (
    df
    .query('population_m > 5')
    .assign(
        log_pop=lambda d: np.log(d.population_m),
        region=lambda d: d.region.astype('category'),
    )
    .groupby('region', observed=True)
    .agg(avg_log=('log_pop', 'mean'), n=('city', 'count'))
    .sort_values('avg_log', ascending=False)
    .reset_index()
)
print(result)


## 10. The first fifteen minutes with any dataset

Run this before forming any opinion. The last loop finds more real problems than
any amount of algorithm selection.


In [ ]:
dirty = pd.DataFrame({
    'country': ['UK', 'U.K.', 'United Kingdom', 'england', 'Uk ', 'UK'],
    'value': [1, 2, 3, 4, 5, 1],
})

print('shape', dirty.shape)
print('dupes', dirty.duplicated().sum())
print()
for c in dirty.select_dtypes(['object', 'string', 'category']):
    print(f'{c}: {dirty[c].nunique()} unique')
    print(dirty[c].value_counts())
print()
print('Five spellings of one country. Your model sees five categories.')


## Try it yourself

1. Normalise the `country` column above (strip, lowercase, map variants) and recount.
2. Build two frames where an inner join silently drops rows, then find them with `indicator=True`.
3. Use `transform` to flag every row more than two standard deviations from its own group mean.
4. Take a frame with duplicate keys and write the three-line guard: count before, merge with
   `validate`, assert after.
